# Comment Category Prediction

## Submission By:

- **Name:** Varun Agnihotri
- **Roll Number:** `24f2004142`

## Import all the required libraries

In [ ]:
import gc
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lightgbm import LGBMClassifier
from scipy.sparse import csr_matrix
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

## Config

In [ ]:
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/comment-category-prediction-challenge")
else:
    DATA_DIR = Path("datasets/")

## Data loading

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_sub = pd.read_csv(
    DATA_DIR / ("Sample.csv" if IS_KAGGLE else "sample_submission.csv")
)

In [ ]:
for _name, _df in [
    ("train", train),
    ("test", test),
    ("sample_sub", sample_sub),
]:
    print(f" {_name} ".center(50, "="))
    print("\nDataset Info:")
    print(_df.info())
    print("\nDataset Description:")
    print(_df.describe(include="all"))
    # print("\nTable Head (10 rows):")
    # print(_df.head(10))
    print("\n\n")

## Helper Functions

In [ ]:
def header(msg: str) -> None:
    """
    Print a formatted header message.
    :param msg: Message to be printed as header.
    :return:
    """
    print("\n" + "=" * 50)
    print(msg.center(50))
    print("=" * 50)

In [ ]:
def clean_text(text: str) -> str:
    """
    Clean the input text by removing URLs, special characters, and extra spaces.
    :param text: Input text string.
    :return: Cleaned text string.
    """
    text = str(text).lower()
    text = re.sub(r"http\\S+|www\\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

In [ ]:
def calc_sparsity(matrix) -> float:
    """
    Calculate the sparsity of a given matrix.
    :param matrix: Input matrix (numpy array or sparse matrix).
    :return: Sparsity in percentage.
    """
    total_elements = matrix.shape[0] * matrix.shape[1]
    non_zero_elements = np.count_nonzero(matrix)
    sparsity = 1.0 - (non_zero_elements / total_elements)
    return sparsity * 100

## Exploratory Data Analysis

In [ ]:
train.shape, test.shape, sample_sub.shape

In [ ]:
train.sample(10)

In [ ]:
# Label distribution
print(train["label"].value_counts())
print(f"\nProportions:\n{train['label'].value_counts(normalize=True)}")

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=train, x="label")
plt.title("Label Distribution")
plt.show()

In [ ]:
train.describe().T

In [ ]:
corr = train.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.show()

In [ ]:
train["comment_length"] = train["comment"].str.len()
train["word_count"] = train["comment"].str.split().str.len()

header("TEXT STATISTICS BY LABEL")
print(train.groupby("label")[["comment_length", "word_count"]].mean())

## Feature Engineering

In [ ]:
def create_features(df):
    df = df.copy()

    # Text features
    df["comment_length"] = df["comment"].str.len()
    df["word_count"] = df["comment"].str.split().str.len()
    df["avg_word_length"] = df["comment"].apply(
        lambda x: np.mean([len(w) for w in str(x).split()]) if str(x).split() else 0
    )

    # Special characters
    df["exclamation_count"] = df["comment"].str.count("!")
    df["question_count"] = df["comment"].str.count("\\?")
    df["capital_ratio"] = df["comment"].apply(
        lambda x: (
            sum(1 for c in str(x) if c.isupper()) / len(str(x))
            if len(str(x)) > 0
            else 0
        )
    )

    # Engagement features
    df["total_votes"] = df["upvote"] + df["downvote"]
    df["vote_ratio"] = np.where(
        df["total_votes"] > 0, df["upvote"] / df["total_votes"], 0.5
    )
    df["vote_diff"] = df["upvote"] - df["downvote"]
    df["log_upvote"] = np.log1p(df["upvote"])
    df["log_downvote"] = np.log1p(df["downvote"])

    # Temporal features
    df["created_date"] = pd.to_datetime(df["created_date"])
    df["hour"] = df["created_date"].dt.hour
    df["day_of_week"] = df["created_date"].dt.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df["is_night"] = ((df["hour"] >= 22) | (df["hour"] <= 6)).astype(int)

    # Emoticon features
    df["total_emoticons"] = df["emoticon_1"] + df["emoticon_2"] + df["emoticon_3"]

    # Post-level aggregations
    post_stats = (
        df.groupby("post_id")
        .agg({"upvote": ["mean", "count"], "downvote": "mean"})
        .reset_index()
    )

    post_stats.columns = [
        "post_id",
        "post_avg_upvote",
        "post_count",
        "post_avg_downvote",
    ]

    df = df.merge(post_stats, on="post_id", how="left")
    df["upvote_vs_post"] = df["upvote"] - df["post_avg_upvote"]
    df["downvote_vs_post"] = df["downvote"] - df["post_avg_downvote"]

    return df

In [ ]:
train = create_features(train)
test = create_features(test)

In [ ]:
print(f"Total Features: {len(train.columns)}")

## Model Building

In [ ]:
train["comment_clean"] = train["comment"].apply(clean_text)
test["comment_clean"] = test["comment"].apply(clean_text)

In [ ]:
# TF-IDF
tfidf = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), min_df=3, max_df=0.9)

In [ ]:
tfidf_train = tfidf.fit_transform(train["comment_clean"])
tfidf_test = tfidf.transform(test["comment_clean"])

In [ ]:
feature_cols = train.select_dtypes(include="number").columns.tolist()
feature_cols.remove("label")

In [ ]:
X_tab = train[feature_cols].fillna(0)
X_tab_test = test[feature_cols].fillna(0)

In [ ]:
scaler = StandardScaler()
X_tab_scaled = scaler.fit_transform(X_tab)
X_tab_test_scaled = scaler.transform(X_tab_test)

In [ ]:
# Combine features
X = np.hstack([tfidf_train.toarray(), X_tab_scaled])
X_sub = np.hstack([tfidf_test.toarray(), X_tab_test_scaled])
y = train["label"].values

print(f"Training shape: {X.shape}")

In [ ]:
# # Cross-validation
# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# n_classes = train["label"].unique().shape[0]
# xgb_preds = np.zeros(len(X_test))
# lgb_preds = np.zeros(len(X_test))
#
# n_splits = skf.get_n_splits()

In [ ]:
# xgb_test_preds = np.zeros((X_test.shape[0], n_classes))
# lgb_test_preds = np.zeros((X_test.shape[0], n_classes))
#
# oof_preds = np.zeros((X_train.shape[0], n_classes))
# for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
#     print(f"Fold {fold + 1}:")
#
#     X_tr, X_val = X_train[tr_idx], X_train[val_idx]
#     y_tr, y_val = y_train[tr_idx], y_train[val_idx]
#
#     # XGBoost
#     xgb = XGBClassifier(
#         n_estimators=300,
#         max_depth=6,
#         early_stopping_rounds=50,
#         learning_rate=0.05,
#         random_state=42,
#     )
#     xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
#     xgb_val_probs = xgb.predict_proba(X_val)
#     xgb_test_preds += xgb.predict_proba(X_test) / n_splits
#
#     # LightGBM
#     lgb = LGBMClassifier(
#         n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42, verbose=-1
#     )
#     lgb.fit(X_tr, y_tr)
#     lgb_val_probs = lgb.predict_proba(X_val)
#     lgb_test_preds += lgb.predict_proba(X_test) / n_splits
#
#     val_ensemble_probs = (0.5 * xgb_val_probs) + (0.5 * lgb_val_probs)
#     val_ensemble_preds = np.argmax(val_ensemble_probs, axis=1)
#     val_acc = accuracy_score(y_val, val_ensemble_preds)
#     print(f"\tAccuracy: {val_acc:.4f}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
models = {
    # Gradient Boosting Models
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        random_state=42,
        objective="multi:softmax",
        num_class=4,
        scale_pos_weight=1,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        random_state=42,
        verbose=-1,
        class_weight="balanced",
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=300,
        max_depth=6,
        learning_rate=0.05,
        random_state=42,
        class_weight="balanced",
    ),
    # Tree-Based Ensemble Models
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1,
    ),
    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=300,
        max_depth=10,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1,
    ),
    # Linear Models
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight="balanced",
        solver="lbfgs",
    ),
    # Neural Network
    "MLP": MLPClassifier(
        hidden_layer_sizes=(128, 64, 32),
        max_iter=500,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
    ),
}

In [ ]:
scores = {}

for model_name, model in models.items():
    header(f"Training {model_name}")
    if model_name == "XGBoost":
        sample_weights = compute_sample_weight("balanced", y_train)
        model.fit(X_train, y_train, sample_weight=sample_weights)
    else:
        model.fit(X_train, y_train)

    val_preds = model.predict(X_test)

    score = f1_score(y_test, val_preds, average="macro")
    scores[model_name] = {
        "score": score,
        "model": model,
    }
    print(f"\t{model_name} Macro F1 Score: {score:.4f}")

In [ ]:
# sort scores
scores = list(sorted(scores.items(), key=lambda x: x[1]["score"], reverse=True))

## Hyperparameter Tuning

In [ ]:
params_dict = {
    # Gradient Boosting Models
    "XGBoost": {
        "n_estimators": [200, 300, 500],
        "max_depth": [4, 6, 8],
        "learning_rate": [0.01, 0.05, 0.1],
        "min_child_weight": [1, 5, 10],
        "subsample": [0.7, 0.8, 0.9],
        "colsample_bytree": [0.7, 0.8, 0.9],
    },
    "LightGBM": {
        "n_estimators": [200, 300, 500],
        "num_leaves": [31, 63, 127],
        "learning_rate": [0.01, 0.05, 0.1],
        "min_child_samples": [10, 20, 50],
        "subsample": [0.7, 0.8, 0.9],
        "colsample_bytree": [0.7, 0.8, 0.9],
        "class_weight": ["balanced", None],
    },
    "HistGradientBoosting": {
        "max_iter": [200, 300, 500],
        "max_depth": [5, 7, 10, None],
        "learning_rate": [0.01, 0.05, 0.1],
        "min_samples_leaf": [10, 20, 50],
        "l2_regularization": [0, 0.1, 1],
        "class_weight": ["balanced", None],
    },
    # Tree-Based Ensemble Models
    "RandomForest": {
        "n_estimators": [200, 300, 500],
        "max_depth": [10, 15, 20, None],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 0.3],
        "class_weight": ["balanced", "balanced_subsample"],
    },
    "ExtraTrees": {
        "n_estimators": [200, 300, 500],
        "max_depth": [10, 15, 20, None],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 0.3],
        "class_weight": ["balanced", "balanced_subsample"],
    },
    # Linear Models
    "LogisticRegression": {
        "C": [0.01, 0.1, 1, 10],
        "class_weight": ["balanced", None],
    },
    # Neural Network
    "MLP": {
        "hidden_layer_sizes": [(64, 32), (128, 64), (128, 64, 32)],
        "alpha": [0.0001, 0.001, 0.01],
        "learning_rate_init": [0.001, 0.01],
        "batch_size": [64, 128, 256],
    },
}

In [ ]:
header("Converting Datasets to Sparse Matrices")

print("Train Dataset:")
print(f"\tSparsity: {calc_sparsity(X_train):.2f}%")
print(f"\tOriginal size: {X_train.nbytes / 1e6:.2f} MB")
X_train = csr_matrix(X_train.astype("float32"))
print(f"\tSparse size: {X_train.data.nbytes / 1e6:.2f} MB")

print("\nTest Dataset:")
print(f"\tSparsity: {calc_sparsity(X_test):.2f}%")
print(f"\tOriginal size: {X_test.nbytes / 1e6:.2f} MB")
X_test = csr_matrix(X_test.astype("float32"))
print(f"\tSparse size: {X_test.data.nbytes / 1e6:.2f} MB")

# Cleanup
gc.collect()

In [ ]:
for model_name, model_data in scores[:3]:
    header(f"Hyperparameter Tuning for {model_name}")

    fit_params = {}
    if model_name == "XGBoost":
        weights = compute_sample_weight("balanced", y_train)
        fit_params["sample_weight"] = weights

    search_cv = RandomizedSearchCV(
        estimator=models[model_name],
        param_distributions=params_dict[model_name],
        n_iter=10,
        scoring="f1_macro",
        cv=5,
        verbose=3,
        random_state=42,
        n_jobs=-1,
    )

    search_cv.fit(X_train, y_train, **fit_params)
    scores.append(
        (
            f"{model_name}_Tuned",
            {
                "score": search_cv.best_score_,
                "model": search_cv.best_estimator_,
            },
        )
    )

    print(f"\t{model_name}_Tuned Best Score: {search_cv.best_score_:.4f}")

In [ ]:
# sort scores
scores = list(sorted(scores.items(), key=lambda x: x[1]["score"], reverse=True))

## Predicting Test Data

In [ ]:
best_model_name, best_model_data = scores[0]

header(f"Best Model: {best_model_name}")
print(f"Macro F1 Score: {best_model_data['score']:.4f}")

print("\nClassification Report:")
best_val_preds = best_model_data["model"].predict(X_test)
print(classification_report(y_test, best_val_preds))

print("\nRetraining on full training data...")
model = best_model_data["model"]

if best_model_name == "XGBoost":
    sample_weights = compute_sample_weight("balanced", y)
    model.fit(X, y, sample_weight=sample_weights)
else:
    model.fit(X, y)

# Final predictions
final_preds = model.predict(X_sub)

## Final Submission

In [ ]:
# Ensemble
# final_test_probs = 0.5 * xgb_test_preds + 0.5 * lgb_test_preds
# final_preds = np.argmax(final_test_probs, axis=1)

submission = pd.DataFrame({"ID": test.index + 1, "label": final_preds})
submission.to_csv("submission.csv", index=False)

header("Submission created!")
print(submission["label"].value_counts())